<a href="https://colab.research.google.com/github/OuhmadMohamed/DI_Bootcamp/blob/main/Week9/Day4/Daily_Chllenge_w9_D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score

# ================= DEVICE =================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ================= PATHS =================
TRAIN_PATH = "/content/train_essays.csv"
TEST_PATH = "/content/test_essays.csv"

src_train = pd.read_csv(TRAIN_PATH)
src_sub = pd.read_csv("/content/sample_submission.csv")

# ================= MODEL =================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased").to(device)
embedding_model = pretrained_model.bert.to(device)

# Freeze BERT
for p in embedding_model.parameters():
    p.requires_grad = False

# ================= PARAMS =================
train_batch_size = 16
test_batch_size = 32
lr = 2e-5
beta1 = 0.5
nz = 100
num_epochs = 3
train_ratio = 0.8

# ================= DATA =================
class Dataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

split = int(len(src_train) * train_ratio)

train_df = src_train.iloc[:split]
val_df = src_train.iloc[split:]

train_loader = DataLoader(
    Dataset(train_df.text.tolist(), train_df.generated.tolist()),
    batch_size=train_batch_size,
    shuffle=True
)

val_loader = DataLoader(
    Dataset(val_df.text.tolist(), val_df.generated.tolist()),
    batch_size=test_batch_size
)

# ================= EMBEDDING =================
def get_embedding(texts):
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

    input_ids = enc["input_ids"].to(device)
    token_type_ids = enc["token_type_ids"].to(device)

    out = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)
    return out.last_hidden_state

# ================= MODELS =================
config = pretrained_model.config # Use the config from the pretrained model

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(nz, 128 * 64)

        self.conv = nn.Sequential(
            nn.Unflatten(1, (128, 64)),
            nn.Conv1d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv1d(256, 768, 3, padding=1),
            nn.ReLU()
        )

        self.encoder = BertEncoder(config)

    def forward(self, z):
        x = self.fc(z)
        x = self.conv(x)
        x = x.permute(0, 2, 1)

        # Pass attention_mask=None to let BertEncoder handle it internally
        out = self.encoder(hidden_states=x, attention_mask=None)
        return out.last_hidden_state


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = BertEncoder(config)
        self.encoder.layer = nn.ModuleList(
            pretrained_model.bert.encoder.layer[:6]
        )

        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        # Pass attention_mask=None to let BertEncoder handle it internally
        out = self.encoder(hidden_states=x, attention_mask=None)
        out = out.last_hidden_state.mean(dim=1)

        return torch.sigmoid(self.classifier(out)).view(-1)

# ================= INIT =================
netG = Generator().to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()

optG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))
optD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))

# ================= TRAIN STEP =================
def train_step(real, labels):
    batch_size = real.size(0)

    # ==== Train D ====
    netD.zero_grad()

    out_real = netD(real)
    loss_real = criterion(out_real, labels)

    noise = torch.randn(batch_size, nz, device=device)
    fake = netG(noise)

    fake_labels = torch.ones(batch_size, device=device)
    out_fake = netD(fake.detach())
    loss_fake = criterion(out_fake, fake_labels)

    loss_D = loss_real + loss_fake
    loss_D.backward()
    optD.step()

    # ==== Train G ====
    netG.zero_grad()

    target = torch.zeros(batch_size, device=device)
    out = netD(fake)
    loss_G = criterion(out, target)

    loss_G.backward()
    optG.step()

# ================= EVAL =================
def evaluate():
    netD.eval()
    preds, y = [], []

    with torch.no_grad():
        for texts, labels in val_loader:
            emb = get_embedding(texts)
            out = netD(emb)

            preds.extend(out.cpu().numpy())
            y.extend(labels.numpy())

    auc = roc_auc_score(y, preds)
    print("AUC:", auc)
    netD.train()
    return auc

# ================= TRAIN LOOP =================
history = []

for epoch in range(num_epochs):
    for texts, labels in train_loader:
        emb = get_embedding(texts)
        labels = labels.float().to(device)

        train_step(emb, labels)

    auc = evaluate()

    history.append({
        "epoch": epoch,
        "state": netD.state_dict(),
        "auc": auc
    })

print("Training done")

# ================= INFERENCE =================
best = max(history, key=lambda x: x["auc"])

model = Discriminator().to(device)
model.load_state_dict(best["state"])
model.eval()

test_df = pd.read_csv(TEST_PATH)

preds = []

with torch.no_grad():
    for i in range(0, len(test_df), 32):
        texts = test_df.text.iloc[i:i+32].tolist()
        emb = get_embedding(texts)

        out = model(emb)
        preds.extend(out.cpu().numpy())

sub = src_sub.copy()
sub["generated"] = preds

print(sub.head())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AUC: 0.9709090909090909
AUC: 0.9818181818181818
AUC: 0.9890909090909091
Training done
         id  generated
0  0000aaaa   0.479745
1  1111bbbb   0.501319
2  2222cccc   0.496220
